In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Często używane parametry transpilacji

<details>
<summary><b>Wersje pakietów</b></summary>

Kod na tej stronie został opracowany przy użyciu poniższych wymagań.
Zalecamy korzystanie z tych lub nowszych wersji.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Ta strona opisuje niektóre z częściej używanych parametrów lokalnej transpilacji. Parametry te konfiguruje się za pomocą argumentów przekazywanych do [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) lub [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile).

<span id="approx-degree"></span>
## Stopień aproksymacji
Możesz użyć stopnia aproksymacji, aby określić, jak dokładnie chcesz, by wynikowy Circuit odpowiadał pożądanemu (wejściowemu) Circuit. Jest to liczba zmiennoprzecinkowa z zakresu (0.0–1.0), gdzie 0.0 oznacza maksymalną aproksymację, a 1.0 (domyślnie) — brak aproksymacji. Mniejsze wartości zamieniają dokładność wyjścia na łatwość wykonania (czyli mniejszą liczbę Gate'ów). Wartość domyślna to 1.0.

W syntezie unitarnej dwu-Qubitowej (stosowanej na początkowych etapach wszystkich poziomów oraz na etapie optymalizacji przy poziomie optymalizacji 3) wartość ta określa docelową wierność dekompozycji wyjściowej. Inaczej mówiąc, jak duży błąd zostaje wprowadzony przy konwersji macierzowej reprezentacji Circuit na dyskretne Gate'y. Jeśli stopień aproksymacji jest niższy (większa aproksymacja), wynikowy Circuit z syntezy będzie się bardziej różnić od macierzy wejściowej, ale prawdopodobnie będzie miał mniej Gate'ów (ponieważ dowolna dwu-Qubitowa operacja może zostać dokładnie zdekomponowana przy użyciu co najwyżej trzech Gate'ów CX) i będzie łatwiejszy do wykonania.

Gdy stopień aproksymacji jest mniejszy niż 1.0, mogą zostać zsyntetyzowane Circuit z jednym lub dwoma Gate'ami CX, co prowadzi do mniejszego błędu sprzętowego kosztem błędu aproksymacji. Ponieważ CX jest najbardziej kosztownym Gate'em pod względem błędu, może być korzystne zmniejszenie ich liczby kosztem wierności w syntezie (technika ta była stosowana w celu zwiększenia quantum volume na urządzeniach IBM&reg;: [Validating quantum computers using randomized model circuits](https://arxiv.org/abs/1811.12926)).

Jako przykład generujemy losowy dwu-Qubitowy `UnitaryGate`, który zostanie zsyntetyzowany na początkowym etapie. Ustawienie `approximation_degree` poniżej 1.0 może wygenerować aproksymowany Circuit. Musimy też podać `basis_gates`, aby metoda syntezy wiedziała, których Gate'ów może użyć do aproksymowanej syntezy.

In [1]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import random_unitary
from qiskit.transpiler import generate_preset_pass_manager

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qubits = QuantumRegister(2, name="q")
qc = QuantumCircuit(qubits)
qc.append(rand_U, qubits)
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    approximation_degree=0.85,
    basis_gates=["sx", "rz", "cx"],
)
approx_qc = pass_manager.run(qc)
print(approx_qc.count_ops()["cx"])

2


This yields an output of `2` because the approximation requires fewer CX gates.

<span id="seed"></span>
## Random number generator seed

Some parts of the transpiler are stochastic, so repeated transpilation runs may return different results. To obtain a reproducible result, you can set the seed for the pseudorandom number generator using the `seed_transpiler` argument. Repeated runs using the same seed will return the same results.

Example:

In [2]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, seed_transpiler=11, basis_gates=["sx", "rz", "cx"]
)
optimized_1 = pass_manager.run(qc)
optimized_1.draw("mpl")

<Image src="../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg" alt="Output of the previous code cell" />

Wynikiem jest `2`, ponieważ aproksymacja wymaga mniejszej liczby Gate'ów CX.

<span id="seed"></span>
## Ziarno generatora liczb losowych
Niektóre części Transpilera są stochastyczne, więc kolejne uruchomienia transpilacji mogą zwracać różne wyniki. Aby uzyskać powtarzalny wynik, możesz ustawić ziarno dla generatora liczb pseudolosowych za pomocą argumentu `seed_transpiler`. Kolejne uruchomienia z tym samym ziarnem będą zwracać te same wyniki.

Przykład:

In [3]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.transpiler import Layout

backend = FakeSherbrooke()

a, b = qubits
initial_layout = Layout({a: 5, b: 6})

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg)

<span id="init-layout"></span>
## Wstępny układ
Przed transpilacją Qubity zawarte w twoim Circuit są wirtualnymi Qubitami, które niekoniecznie odpowiadają fizycznym Qubitom na docelowym Backend. Możesz określić wstępne mapowanie wirtualnych Qubitów na fizyczne Qubity za pomocą argumentu `initial_layout`. Pamiętaj, że końcowy układ Qubitów może różnić się od wstępnego, ponieważ Transpiler może przestawiać Qubity przy użyciu bramek swap lub innych metod.

W poniższym przykładzie konstruujemy wstępny układ dla atrapowego Backend [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke), tworząc obiekt [`Layout`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Layout). Nasz układ mapuje pierwszy Qubit naszego Circuit na Qubit 5 Sherbrooke, a drugi Qubit naszego Circuit na Qubit 6 Sherbrooke. Zauważ, że fizyczne Qubity są zawsze reprezentowane przez liczby całkowite.

In [4]:
initial_layout = [5, 6]

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg)

Oprócz podawania obiektu Layout możesz też przekazać listę liczb całkowitych, gdzie $i$-ty element listy zawiera fizyczny Qubit, na który powinien zostać zmapowany $i$-ty Qubit. Na przykład:

In [5]:
from qiskit.visualization import plot_error_map

plot_error_map(backend, figsize=(30, 24))

<Image src="../docs/images/guides/common-parameters/extracted-outputs/8df57c6a-1ff4-4d58-9b7e-4378452c3025-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg)

Możesz użyć funkcji [`plot_error_map`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.plot_error_map), aby wygenerować diagram grafu urządzenia z informacjami o błędach i oznaczonymi fizycznymi Qubitami. Podobne diagramy możesz też przeglądać na stronie [Compute resources](https://quantum.cloud.ibm.com/computers).